In [0]:
from functools import reduce
from pyspark.sql import functions as F
from pyspark.sql.window import Window

B = "workspace.bronze"
S = "workspace.silver"
G = "workspace.gold"

In [0]:
DIM = f"{G}.dim_client"
STG = f"{G}.stg_dim_client"

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {DIM} (
    client_sk            BIGINT,
    client_id            INT,
    nric_hash            STRING,
    full_name_masked     STRING,
    address              STRING,
    region               STRING,
    household_income     DECIMAL(10,2),
    household_size       INT,
    income_band          STRING,
    effective_start_date DATE,
    effective_end_date   DATE,
    is_current           BOOLEAN,
    source_file          STRING
)
""")

TRACKED = ["address", "region", "household_income", "household_size"]

In [0]:
""" Create client dimension table """
def load_dim_client(source_file):
    inc = spark.table(f"{S}.clients").filter(F.col("source_file") == source_file).alias("i")
    cur = spark.table(DIM).filter("is_current = true").alias("c")
 
    # Join incoming with current on client_id
    joined = inc.join(cur, F.col("i.client_id") == F.col("c.client_id"), "left")
    # Check for which rows are new or changed
    same = reduce(lambda a, b: a & b,
                  [F.col(f"i.{c}").eqNullSafe(F.col(f"c.{c}")) for c in TRACKED])
    # Check for which rows are new
    is_new = F.col("c.client_sk").isNull()
 
    # Find new clients and changed clients, and stage them
    (joined.filter(is_new | ~same).select(
        F.col("i.client_id").alias("client_id"),
        F.col("i.nric_hash").alias("nric_hash"),
        F.col("i.full_name_masked").alias("full_name_masked"),
        F.col("i.address").alias("address"),
        F.col("i.region").alias("region"),
        F.col("i.household_income").alias("household_income"),
        F.col("i.household_size").alias("household_size"),
        F.col("i.updated_date").alias("src_date"),
        is_new.alias("is_new"),
        F.col("i.source_file").alias("source_file"))
     .write.mode("overwrite").saveAsTable(STG))
 
    stg = spark.table(STG)
    n_new = stg.filter("is_new = true").count()
    n_changed = stg.filter("is_new = false").count()
    if n_new + n_changed == 0:
        print(f"{source_file}: no new or changed clients, nothing to do")
        spark.sql(f"DROP TABLE IF EXISTS {STG}")
        return
 
    # Close the current version of changed clients
    spark.sql(f"""
        MERGE INTO {DIM} t
        USING (SELECT client_id, src_date FROM {STG} WHERE is_new = false) s
        ON t.client_id = s.client_id AND t.is_current = true
        WHEN MATCHED THEN UPDATE SET
            t.is_current = false,
            t.effective_end_date = date_sub(s.src_date, 1)
    """)
 
    max_sk = spark.sql(f"SELECT coalesce(max(client_sk), 0) FROM {DIM}").collect()[0][0]
    new_rows = (
        stg
        # create a new identifier
        .withColumn("client_sk",
                    (F.row_number().over(Window.orderBy("client_id")) + F.lit(max_sk)).cast("bigint"))
        # create categories for income
        .withColumn("income_band",
                    F.when(F.col("household_income") < 2000, "Below 2000")
                     .when(F.col("household_income") < 4000, "2000-3999")
                     .otherwise("4000+"))
        # Defining which period of the data is effective
        .withColumn(
            "effective_start_date",
            F.when(
                F.col("is_new") == True,
                F.to_date(F.lit("1900-01-01"))
            ).otherwise(
                F.col("src_date")
            )
        )
        .withColumn("effective_end_date", F.to_date(F.lit("9999-12-31")))
        .withColumn("is_current", F.lit(True))
        .select("client_sk", "client_id", "nric_hash", "full_name_masked", "address", "region",
                "household_income", "household_size", "income_band",
                "effective_start_date", "effective_end_date", "is_current", "source_file")
    )
    new_rows.write.mode("append").saveAsTable(DIM)
 
    spark.sql(f"DROP TABLE IF EXISTS {STG}")
    print(f"{source_file}: {n_new} new clients, {n_changed} changed clients (old version closed, new inserted)")
 
 
load_dim_client("clients_day1.csv")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


clients_day1.csv: 982 new clients, 0 changed clients (old version closed, new inserted)


In [0]:
""" Create scheme dimension table """
dim_scheme = spark.table(f"{S}.schemes").select("scheme_id", "scheme_name", "category")
dim_scheme.write.mode("overwrite").saveAsTable(f"{G}.dim_scheme")

In [0]:
""" Create date dimension table """
def build_dim_date():
    b = spark.table(f"{S}.assistance").agg(
        F.min("application_date").alias("mn"), F.max("application_date").alias("mx")).collect()[0]
    # generate all the dates between max and min dates
    dates = spark.sql(
        f"SELECT explode(sequence(to_date('{b['mn']}'), to_date('{b['mx']}'), interval 1 day)) AS date")
    (dates
     .withColumn("date_key", F.date_format("date", "yyyyMMdd").cast("int"))
     .withColumn("year", F.year("date"))
     .withColumn("quarter", F.quarter("date"))
     .withColumn("month", F.month("date"))
     .withColumn("month_name", F.date_format("date", "MMMM"))
     .withColumn("day_of_week", F.date_format("date", "EEEE"))
     .select("date_key", "date", "year", "quarter", "month", "month_name", "day_of_week")
     .write.mode("overwrite").saveAsTable(f"{G}.dim_date"))
 
build_dim_date()

In [0]:
""" Create fact table """
def build_fact():
    a = spark.table(f"{S}.assistance").alias("a")
    c = spark.table(DIM).alias("c")

    fact = (a.join(c,
                   (F.col("a.client_id") == F.col("c.client_id")) &
                   F.col("a.application_date").between(F.col("c.effective_start_date"), F.col("c.effective_end_date")), "left")
            .select(F.col("a.case_id").alias("case_id"),
                    F.col("c.client_sk").alias("client_sk"),
                    F.col("a.scheme_id").alias("scheme_id"),
                    F.date_format("a.application_date", "yyyyMMdd").cast("int").alias("date_key"),
                    F.col("a.assistance_amount").alias("assistance_amount"),
                    F.col("a.status").alias("status")))
    fact.write.mode("overwrite").saveAsTable(f"{G}.fact_assistance")
 
    total = spark.table(f"{G}.fact_assistance").count()
    unmatched = spark.table(f"{G}.fact_assistance").filter("client_sk IS NULL").count()
    print(f"fact_assistance: {total} rows, {unmatched} without a client version (should be 0)")
 
 
build_fact()

fact_assistance: 2355 rows, 0 without a client version (should be 0)


In [0]:
# How much assistance by region and income band? (star schema query)
display(spark.sql(f"""
    SELECT c.region, c.income_band, round(sum(f.assistance_amount), 2) AS total_assistance
    FROM {G}.fact_assistance f JOIN {G}.dim_client c ON f.client_sk = c.client_sk
    WHERE f.status IN ('Approved', 'Disbursed')
    GROUP BY c.region, c.income_band ORDER BY c.region, c.income_band
"""))
 
# SCD2 proof: clients with more than one version (only appears after Day 2 is loaded)
display(spark.sql(f"""
    SELECT client_id, address, region, effective_start_date, effective_end_date, is_current
    FROM {DIM}
    WHERE client_id IN (SELECT client_id FROM {DIM} GROUP BY client_id HAVING count(*) > 1)
    ORDER BY client_id, effective_start_date
    LIMIT 20
"""))

region,income_band,total_assistance
Central,2000-3999,108515.71
Central,4000+,93555.13
Central,Below 2000,94274.25
East,2000-3999,68945.94
East,4000+,93428.35
East,Below 2000,109523.14
North,2000-3999,91545.77
North,4000+,86246.80
North,Below 2000,71388.52
North-East,2000-3999,100292.24


client_id,address,region,effective_start_date,effective_end_date,is_current
5,"Flat 7, Gemma Common",East,1900-01-01,2026-10-01,false
5,"Studio 98M, Smith Harbor",West,2026-10-02,9999-12-31,true
19,301 Bernard Cliff,Central,1900-01-01,2026-10-01,false
19,"Studio 04h, Evans Keys",North,2026-10-02,9999-12-31,true
22,6 Roberts Coves,North,1900-01-01,2026-10-01,false
22,6 Whittaker Squares,Central,2026-10-02,9999-12-31,true
38,02 Gill Loop,Central,1900-01-01,2026-10-01,false
38,9 Booth Curve,Central,2026-10-02,9999-12-31,true
48,0 Duncan Mountains,East,1900-01-01,2026-10-01,false
48,3 Thomas Locks,West,2026-10-02,9999-12-31,true


In [0]:
# add client day 2 to gold dimension table
load_dim_client("clients_day2.csv")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


clients_day2.csv: 38 new clients, 50 changed clients (old version closed, new inserted)


In [0]:
build_fact()

fact_assistance: 2444 rows, 0 without a client version (should be 0)


In [0]:
def snapshot_metrics(label):
    """Store key counts under a label such as '1_after_day1' or '2_after_day2'."""
    B, S, G = "workspace.bronze", "workspace.silver", "workspace.gold"
 
    def q(sql):
        return spark.sql(sql).collect()[0][0]
 
    metrics = {
        "bronze_clients_rows":        q(f"SELECT count(*) FROM {B}.clients"),
        "silver_clients_rows":        q(f"SELECT count(*) FROM {S}.clients"),
        "quarantine_clients_open":    q(f"SELECT count(*) FROM {S}.quarantine_clients WHERE dq_status = 'OPEN'"),
        "silver_assistance_rows":     q(f"SELECT count(*) FROM {S}.assistance"),
        "quarantine_assistance_open": q(f"SELECT count(*) FROM {S}.quarantine_assistance WHERE dq_status = 'OPEN'"),
        "orphan_rows_open":           q(f"SELECT count(*) FROM {S}.quarantine_assistance "
                                        f"WHERE dq_status = 'OPEN' AND reason = 'ORPHAN_CLIENT'"),
        "dim_client_rows":            q(f"SELECT count(*) FROM {G}.dim_client"),
        "dim_client_current_rows":    q(f"SELECT count(*) FROM {G}.dim_client WHERE is_current"),
        "clients_with_history":       q(f"SELECT count(*) FROM (SELECT client_id FROM {G}.dim_client "
                                        f"GROUP BY client_id HAVING count(*) > 1)"),
        "fact_assistance_rows":       q(f"SELECT count(*) FROM {G}.fact_assistance"),
    }
 
    table = f"{G}.pipeline_metrics"
    if spark.catalog.tableExists(table):
        spark.sql(f"DELETE FROM {table} WHERE run_label = '{label}'")
    (spark.createDataFrame([(label, k, int(v)) for k, v in metrics.items()],
                           "run_label STRING, metric STRING, value BIGINT")
          .withColumn("captured_at", F.current_timestamp())
          .write.mode("append").saveAsTable(table))
    print(f"saved {len(metrics)} metrics as '{label}'")
 
 
def show_before_after():
    display(spark.table("workspace.gold.pipeline_metrics")
            .groupBy("metric").pivot("run_label").agg(F.first("value")))

In [0]:
snapshot_metrics("1_after_day1")

saved 10 metrics as '1_after_day1'


In [0]:
snapshot_metrics("2_after_day2")

saved 10 metrics as '2_after_day2'


In [0]:
show_before_after()

metric,1_after_day1,2_after_day2
orphan_rows_open,101,12
fact_assistance_rows,2355,2444
quarantine_clients_open,18,10
silver_clients_rows,982,1992
silver_assistance_rows,2355,2444
bronze_clients_rows,1005,2025
dim_client_rows,982,1070
quarantine_assistance_open,145,56
dim_client_current_rows,982,1020
clients_with_history,0,50


In [0]:
# For visualisation purpose 
def try_sql(stmt):
    try:
        spark.sql(stmt)
    except Exception as e:
        print(f"skipped (probably already exists): {str(e)[:120]}")
 
 
for tbl, col in [("dim_client", "client_sk"), ("dim_scheme", "scheme_id"),
                 ("dim_date", "date_key"), ("fact_assistance", "case_id")]:
    try_sql(f"ALTER TABLE {G}.{tbl} ALTER COLUMN {col} SET NOT NULL")
    try_sql(f"ALTER TABLE {G}.{tbl} ADD CONSTRAINT pk_{tbl} PRIMARY KEY ({col})")
 
try_sql(f"ALTER TABLE {G}.fact_assistance ADD CONSTRAINT fk_fact_client "
        f"FOREIGN KEY (client_sk) REFERENCES {G}.dim_client(client_sk)")
try_sql(f"ALTER TABLE {G}.fact_assistance ADD CONSTRAINT fk_fact_scheme "
        f"FOREIGN KEY (scheme_id) REFERENCES {G}.dim_scheme(scheme_id)")
try_sql(f"ALTER TABLE {G}.fact_assistance ADD CONSTRAINT fk_fact_date "
        f"FOREIGN KEY (date_key) REFERENCES {G}.dim_date(date_key)")

In [0]:
%skip
spark.sql(f"DROP TABLE IF EXISTS {G}.fact_assistance")
spark.sql(f"DROP TABLE IF EXISTS {G}.dim_client")
spark.sql(f"DROP TABLE IF EXISTS {G}.dim_date")